# RangeFactor

A `RangeFactor` models scalar distance measurements between points, poses, and cameras. The related `RangeFactorWithTransform` and `RangeFactorWithTransformBias` classes account for sensor offsets and additive range bias.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sam/doc/RangeFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

from gtsam.symbol_shorthand import B, L, X

## Create a direct range factor

Start with a planar pose, a landmark, and the distance between them. A `RangeFactor2D` stores that scalar measurement and connects the two keys. The inherited `unwhitenedError()` method lets us verify that the measurement agrees with a set of values.

In [3]:
pose_key, landmark_key = X(0), L(0)
pose = gtsam.Pose2(1.0, 2.0, 0.3)
landmark = np.array([4.0, 5.0])
measured_range = pose.range(landmark)
range_noise = gtsam.noiseModel.Isotropic.Sigma(1, 0.1)

factor = gtsam.RangeFactor2D(
    pose_key, landmark_key, measured_range, range_noise
)
values = gtsam.Values()
values.insert(pose_key, pose)
values.insert(landmark_key, landmark)

assert np.allclose(factor.unwhitenedError(values), 0.0)
print("stored measurement:", factor.measured())
print("factor keys:", factor.keys())

stored measurement: 4.242640687119285
factor keys: [8646911284551352320, 7782220156096217088]


## Add a sensor transform and bias

Often the range sensor is not located at the body-frame origin. `RangeFactorWithTransform2D` accepts the fixed body-to-sensor transform and predicts distance from that sensor pose. If the sensor also has an estimated additive offset, `RangeFactorWithTransformBias2D` connects a third key containing the scalar bias.

In [4]:
body_T_sensor = gtsam.Pose2(0.2, 0.0, 0.1)
sensor_pose = pose.compose(body_T_sensor)
sensor_range = sensor_pose.range(landmark)

offset_factor = gtsam.RangeFactorWithTransform2D(
    pose_key, landmark_key, sensor_range, range_noise, body_T_sensor
)
assert np.allclose(offset_factor.unwhitenedError(values), 0.0)

bias_key, bias = B(0), 0.25
bias_factor = gtsam.RangeFactorWithTransformBias2D(
    pose_key,
    landmark_key,
    bias_key,
    sensor_range + bias,
    range_noise,
    body_T_sensor,
)
values.insert(bias_key, bias)
assert np.allclose(bias_factor.unwhitenedError(values), 0.0)
print("offset measurement:", offset_factor.measured())
print("biased measurement:", bias_factor.measured())

offset measurement: 4.066813490320851
biased measurement: 4.316813490320851


## When to use it

Use a direct `RangeFactor` when the measurement origin coincides with the estimated pose or point. Use `RangeFactorWithTransform` for a known sensor offset, and `RangeFactorWithTransformBias` when the graph should estimate an additive range bias. The 3D and camera variants follow the same construction pattern shown above.

For direction-only or combined measurements, continue with [`BearingFactor`](BearingFactor.ipynb) and [`BearingRangeFactor`](BearingRangeFactor.ipynb). [`QuadraticRangeFactor`](QuadraticRangeFactor.ipynb) presents the lifted formulation used by certifiable solvers.

## Source

[`RangeFactor.h`](../RangeFactor.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.